# README

- **Author**: `詹明翰`
- **Created At**: `2026-11-06`
- **Last Modified At**: `2026-11-06`

---

## What does this file do?

- 第一階段稽核夜間事故分群輸入；第二階段依 2026-11-06 學生確認的規則建立正式輸入及兩份模型矩陣。
- 觀察單位暫依既有資料定義為一列一件事故；以 `is_night == 1` 篩選主要夜間樣本。
- 主分析排除三筆關鍵類別欄位缺失及 91 筆速限為 0 的紀錄；低頻類別完整保留。速限 0 不解讀為真實 0 km/h，後續可另作敏感度分析。
- 分群候選輸入為速限、事故小時與月份的週期轉換、天候、道路類別、道路型態及事故位置；照明、傷亡、行政區與座標不得作為分群輸入。

---

## What does this file take?

- **Source Data Sets**:
  1. `data/processed/taipei_2025_descriptive.csv`
     - Input key: `descriptive`
     - Description: 2025 年臺北市死傷交通事故衍生資料，22,762 列、58 欄；`record_id` 為目前資料內唯一鍵。
     - Documentation: `data/processed/taipei_2025_descriptive_documentation.ipynb`
     - SHA-256: `e98fb4c4d62bd08440d58bbe17af7f3c03b48b4bb91e02bf306e48dda8250465`（2026-11-06 實際磁碟版本；與 `PROJECT.md` 舊紀錄 `09c8...` 不同）

---

## What does this file output?

- `data/processed/taipei_2025_nighttime_clustering_input.csv`
  - Output key: `nighttime_clustering_input`
  - Description: 夜間事故分群用、保留稽核識別欄位及已確認模型欄位的分析資料。
  - Role: `analysis`
  - Documentation: `data/processed/taipei_2025_nighttime_clustering_input_documentation.ipynb`（尚未建立）
  - Expected schema and row-count rules: 6,701 列；`record_id` 唯一；模型候選欄位無缺失。
  - Verification status / SHA-256: 重新讀檔檢查通過；`02869cf20f9362a6bf3d54ca2ca8e4b75dca983069d220bc527d6dbe9122b0f1`。
  - Downstream finding input: `data/processed/taipei_2025_nighttime_clustering_input.csv`
- `data/processed/taipei_2025_nighttime_clustering_matrix_unweighted.csv`
  - Output key: `matrix_unweighted`；Role: `analysis`。
  - Description: 數值欄標準化、類別欄一般 one-hot 的 PCA／分群候選矩陣。
  - Expected schema and row-count rules: 6,701 列；`record_id` 加 49 個模型欄位；不得含結果變數。
  - Verification status / SHA-256: 重新讀檔檢查通過；`6a6d92299d333cce4dcc9b1f425faa28e0d8a20ba2b76dc7221ecc1d85328ab6`。
- `data/processed/taipei_2025_nighttime_clustering_matrix_block_weighted.csv`
  - Output key: `matrix_block_weighted`；Role: `analysis`。
  - Description: 數值欄標準化，並將每個 one-hot 類別區塊除以該區塊類別數平方根的比較矩陣。
  - Expected schema and row-count rules: 與一般矩陣相同；僅類別區塊權重不同。
  - Verification status / SHA-256: 重新讀檔檢查通過；`88173998c8c597af0ce653e1b6764b03ed2582cdb961c3975de12079ab304e37`。

所有路徑均相對於含 `AGENTS.md` 與 `templates/` 的學生專案根目錄。


In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'templates').is_dir():
            return candidate
    raise FileNotFoundError('請從含 AGENTS.md 與 templates/ 的學生專案內開啟本 Notebook。')


def project_file(relative_path, area, must_exist=True):
    rel = Path(relative_path)
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError(f'必須使用明確的專案相對路徑：{relative_path}')
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / 'data' / area).resolve()
    if not path.is_relative_to(allowed):
        raise ValueError(f'檔案必須位於 data/{area}/：{relative_path}')
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


PROJECT_ROOT = find_project_root()


In [2]:
INPUTS = {'descriptive': 'data/processed/taipei_2025_descriptive.csv'}
EXPECTED_INPUT_SHA256 = {
    'descriptive': 'e98fb4c4d62bd08440d58bbe17af7f3c03b48b4bb91e02bf306e48dda8250465'
}
OUTPUTS = {
    'nighttime_clustering_input': 'data/processed/taipei_2025_nighttime_clustering_input.csv'
    ,'matrix_unweighted': 'data/processed/taipei_2025_nighttime_clustering_matrix_unweighted.csv'
    ,'matrix_block_weighted': 'data/processed/taipei_2025_nighttime_clustering_matrix_block_weighted.csv'
}

input_path = project_file(INPUTS['descriptive'], 'processed')
actual_sha256 = sha256_file(input_path)
if actual_sha256 != EXPECTED_INPUT_SHA256['descriptive']:
    raise ValueError('輸入檔版本已改變；請先辨識新版本，不可靜默改用。')

output_paths = {key: project_file(path, 'processed', must_exist=False) for key, path in OUTPUTS.items()}
assert len(set(output_paths.values())) == len(output_paths)
assert input_path not in set(output_paths.values()), '輸出不得覆寫輸入。'

df = pd.read_csv(input_path, encoding='utf-8-sig', low_memory=False)
assert df.shape == (22762, 58), f'非預期資料尺寸：{df.shape}'
assert df.columns.is_unique
assert df['record_id'].is_unique and df['record_id'].notna().all()
required_derived = {'record_id', 'crash_datetime', 'district', 'is_night', 'lighting_label', 'fatal24'}
missing_required = required_derived - set(df.columns)
assert not missing_required, f'缺少必要衍生欄位：{sorted(missing_required)}'
print({'input': INPUTS['descriptive'], 'rows': len(df), 'columns': len(df.columns),
       'sha256': actual_sha256, 'record_id_unique': df['record_id'].is_unique})


{'input': 'data/processed/taipei_2025_descriptive.csv', 'rows': 22762, 'columns': 58, 'sha256': 'e98fb4c4d62bd08440d58bbe17af7f3c03b48b4bb91e02bf306e48dda8250465', 'record_id_unique': True}


## 可稽核欄位對應

以下以明確中文欄名讀取，並同時核對輸入雜湊、58 欄結構及既有官方欄序。天候為第 13 欄、道路類別第 15 欄、速限第 16 欄、道路型態第 17 欄、事故位置第 18 欄（Python 位置 12、14、15、16、17）；這些檢查不改變原始值。


In [3]:
SOURCE_FIELDS = {
    'weather_code': ('天候', 12),
    'road_class_code': ('道路類別', 14),
    'speed_limit': ('速限-速度限制', 15),
    'road_form_code': ('道路型態', 16),
    'crash_position_code': ('事故位置', 17),
}

source_headers = list(df.columns[:47])
assert list(df.columns[47:]) == [
    'record_id', 'crash_datetime', 'district', 'lighting_label', 'weather_label',
    'road_form_label', 'fatal24', 'night_fixed', 'sunrise_local', 'sunset_local', 'is_night'
]

audit_df = df.copy()
for new_name, (source_name, position) in SOURCE_FIELDS.items():
    assert source_name in df.columns, f'缺少原始欄位：{source_name}'
    assert df.columns[position] == source_name, f'{source_name} 的欄序與已核對結構不同。'
    audit_df[new_name] = pd.to_numeric(audit_df[source_name], errors='coerce')

field_mapping = pd.DataFrame([
    {'analysis_name': name, 'source_name': source_name, 'python_position': position,
     'human_column_number': position + 1}
    for name, (source_name, position) in SOURCE_FIELDS.items()
])
display(field_mapping)


,analysis_name,source_name,python_position,human_column_number
0,weather_code,天候,12,13
1,road_class_code,道路類別,14,15
2,speed_limit,速限-速度限制,15,16
3,road_form_code,道路型態,16,17
4,crash_position_code,事故位置,17,18


In [4]:
night = audit_df.loc[audit_df['is_night'].eq(1)].copy()
night['crash_datetime'] = pd.to_datetime(night['crash_datetime'], errors='coerce')
assert len(night) == 6795
assert night['crash_datetime'].notna().all()

night['hour_sin'] = np.sin(2 * np.pi * night['crash_datetime'].dt.hour / 24)
night['hour_cos'] = np.cos(2 * np.pi * night['crash_datetime'].dt.hour / 24)
night['month_sin'] = np.sin(2 * np.pi * (night['crash_datetime'].dt.month - 1) / 12)
night['month_cos'] = np.cos(2 * np.pi * (night['crash_datetime'].dt.month - 1) / 12)

NUMERIC_MODEL_FIELDS = ['speed_limit', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']
CATEGORICAL_MODEL_FIELDS = ['weather_code', 'road_class_code', 'road_form_code', 'crash_position_code']
MODEL_FIELDS = NUMERIC_MODEL_FIELDS + CATEGORICAL_MODEL_FIELDS
EXCLUDED_FROM_CLUSTERING = [
    'lighting_label', 'fatal24', 'district', 'is_night', 'night_fixed',
    'sunrise_local', 'sunset_local'
]
assert not set(MODEL_FIELDS) & set(EXCLUDED_FROM_CLUSTERING)
assert not any(token in column.lower() for column in MODEL_FIELDS
               for token in ['light', 'fatal', 'death', 'injury', 'district', 'longitude', 'latitude'])
print({'all_rows': len(df), 'night_rows': len(night),
       'candidate_model_fields': MODEL_FIELDS,
       'explicitly_excluded': EXCLUDED_FROM_CLUSTERING})


{'all_rows': 22762, 'night_rows': 6795, 'candidate_model_fields': ['speed_limit', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'weather_code', 'road_class_code', 'road_form_code', 'crash_position_code'], 'explicitly_excluded': ['lighting_label', 'fatal24', 'district', 'is_night', 'night_fixed', 'sunrise_local', 'sunset_local']}


In [5]:
input_audit = pd.DataFrame({
    'dtype': night[MODEL_FIELDS].dtypes.astype(str),
    'missing_n': night[MODEL_FIELDS].isna().sum(),
    'missing_pct': night[MODEL_FIELDS].isna().mean().mul(100),
    'unique_nonmissing': night[MODEL_FIELDS].nunique(dropna=True),
}).rename_axis('field').reset_index()
display(input_audit)

complete_case_mask = night[MODEL_FIELDS].notna().all(axis=1)
missing_record_ids = night.loc[~complete_case_mask, ['record_id'] + MODEL_FIELDS]
print({'night_rows': len(night), 'complete_case_rows': int(complete_case_mask.sum()),
       'rows_with_any_candidate_input_missing': int((~complete_case_mask).sum())})
display(missing_record_ids)


,field,dtype,missing_n,missing_pct,unique_nonmissing
0,speed_limit,float64,0,0.00000,9
1,hour_sin,float64,0,0.00000,14
2,hour_cos,float64,0,0.00000,13
3,month_sin,float64,0,0.00000,11
4,month_cos,float64,0,0.00000,11
5,weather_code,float64,0,0.00000,4
6,road_class_code,float64,3,0.04415,5
7,road_form_code,float64,3,0.04415,14
8,crash_position_code,float64,3,0.04415,23


{'night_rows': 6795, 'complete_case_rows': 6792, 'rows_with_any_candidate_input_missing': 3}


,record_id,speed_limit,hour_sin,hour_cos,month_sin,month_cos,weather_code,road_class_code,road_form_code,crash_position_code
501,502,50.0,-0.965926,-0.258819,0.000000,1.0,7.0,NaN,NaN,NaN
9071,9072,50.0,-0.965926,0.258819,0.866025,-0.5,8.0,NaN,NaN,NaN
15263,15264,50.0,-0.500000,0.866025,-0.866025,-0.5,7.0,NaN,NaN,NaN


In [6]:
def category_frequency(frame, column):
    counts = frame[column].value_counts(dropna=False).rename('n').reset_index()
    counts.columns = ['value', 'n']
    counts.insert(0, 'field', column)
    counts['pct_of_night_sample'] = counts['n'].div(len(frame)).mul(100)
    counts['below_1_pct'] = counts['pct_of_night_sample'].lt(1)
    return counts

category_audit = pd.concat(
    [category_frequency(night, column) for column in CATEGORICAL_MODEL_FIELDS],
    ignore_index=True,
)
display(category_audit)

speed_limit_audit = category_frequency(night, 'speed_limit')
display(speed_limit_audit)

observed_one_hot_columns = int(sum(night[column].nunique(dropna=True)
                                   for column in CATEGORICAL_MODEL_FIELDS))
print({
    'speed_limit_zero_n': int(night['speed_limit'].eq(0).sum()),
    'speed_limit_zero_pct': float(night['speed_limit'].eq(0).mean() * 100),
    'observed_categorical_levels_total': observed_one_hot_columns,
    'estimated_columns_without_missing_indicators': observed_one_hot_columns + len(NUMERIC_MODEL_FIELDS),
    'rare_levels_below_1_pct_n': int(category_audit['below_1_pct'].sum()),
})


,field,value,n,pct_of_night_sample,below_1_pct
0,weather_code,8.0,3063,45.077263,False
1,weather_code,7.0,2490,36.644592,False
2,weather_code,6.0,1241,18.263429,False
3,weather_code,9.0,1,0.014717,True
4,road_class_code,5.0,6538,96.217807,False
5,road_class_code,8.0,116,1.707138,False
6,road_class_code,6.0,85,1.250920,False
7,road_class_code,7.0,49,0.721118,True
8,road_class_code,4.0,4,0.058867,True
9,road_class_code,NaN,3,0.044150,True


,field,value,n,pct_of_night_sample,below_1_pct
0,speed_limit,50.0,4675,68.800589,False
1,speed_limit,30.0,1253,18.440029,False
2,speed_limit,40.0,681,10.022075,False
3,speed_limit,0.0,91,1.339220,False
4,speed_limit,60.0,54,0.794702,True
5,speed_limit,70.0,22,0.323767,True
6,speed_limit,80.0,9,0.132450,True
7,speed_limit,20.0,7,0.103017,True
8,speed_limit,25.0,3,0.044150,True


{'speed_limit_zero_n': 91, 'speed_limit_zero_pct': 1.3392200147167035, 'observed_categorical_levels_total': 46, 'estimated_columns_without_missing_indicators': 51, 'rare_levels_below_1_pct_n': 29}


## 第二階段：套用已確認規則並建立正式輸入

2026-11-06 學生回覆「照建議」，確認以下主分析規則：

1. 排除三筆道路類別、道路型態或事故位置缺失的事故。
2. 主分析排除 91 筆 `speed_limit == 0`；不把它們解讀成真實 0 km/h，也不任意補值。
3. 低頻類別全部保留；不只因低於 1% 就合併。
4. 同時建立一般 one-hot 與類別區塊加權矩陣，供第三階段比較；這不預先決定哪一版產生較好的分群。

正式輸入保留可讀的原始代碼與週期變數；兩份矩陣均保留 `record_id` 作連結鍵。照明、傷亡、行政區及座標均不寫入模型矩陣。


In [7]:
analysis_mask = complete_case_mask & night['speed_limit'].ne(0)
prepared = night.loc[analysis_mask].copy()
assert len(prepared) == 6701
assert int((~complete_case_mask).sum()) == 3
assert int(night['speed_limit'].eq(0).sum()) == 91
assert not (night.loc[~complete_case_mask, 'speed_limit'].eq(0)).any(), (
    '排除原因重疊，列數說明需重新核對。'
)

semantic_columns = [
    'record_id', 'crash_datetime', 'speed_limit',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'weather_code', 'road_class_code', 'road_form_code', 'crash_position_code',
]
semantic = prepared[semantic_columns].copy()
semantic['crash_datetime'] = semantic['crash_datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
for column in ['record_id', 'speed_limit'] + CATEGORICAL_MODEL_FIELDS:
    semantic[column] = semantic[column].astype('int64')
assert semantic['record_id'].is_unique
assert not semantic.isna().any().any()

exclusion_audit = pd.DataFrame([
    {'rule': 'is_night == 1', 'excluded_n': len(df) - len(night), 'remaining_n': len(night)},
    {'rule': '關鍵模型欄位完整', 'excluded_n': int((~complete_case_mask).sum()),
     'remaining_n': int(complete_case_mask.sum())},
    {'rule': 'speed_limit != 0', 'excluded_n': int(night['speed_limit'].eq(0).sum()),
     'remaining_n': len(prepared)},
])
display(exclusion_audit)
display(semantic.head(10))


,rule,excluded_n,remaining_n
0,is_night == 1,15967,6795
1,關鍵模型欄位完整,3,6792
2,speed_limit != 0,91,6701


,record_id,crash_datetime,speed_limit,hour_sin,hour_cos,month_sin,month_cos,weather_code,road_class_code,road_form_code,crash_position_code
0,1,2025-01-01 00:18:00,40,0.000000,1.000000e+00,0.0,1.0,8,5,14,9
1,2,2025-01-01 00:31:00,50,0.000000,1.000000e+00,0.0,1.0,7,5,14,9
2,3,2025-01-01 00:35:00,50,0.000000,1.000000e+00,0.0,1.0,7,5,5,1
3,4,2025-01-01 00:41:00,50,0.000000,1.000000e+00,0.0,1.0,7,5,14,7
4,5,2025-01-01 00:57:00,50,0.000000,1.000000e+00,0.0,1.0,7,5,3,1
5,6,2025-01-01 01:34:00,30,0.258819,9.659258e-01,0.0,1.0,7,5,3,1
6,7,2025-01-01 02:15:00,50,0.500000,8.660254e-01,0.0,1.0,7,5,14,9
7,8,2025-01-01 06:26:00,30,1.000000,6.123234e-17,0.0,1.0,8,5,12,9
8,9,2025-01-01 06:29:00,50,1.000000,6.123234e-17,0.0,1.0,8,5,3,2
36,37,2025-01-01 17:40:00,50,-0.965926,-2.588190e-01,0.0,1.0,6,5,5,1


In [8]:
numeric = semantic[NUMERIC_MODEL_FIELDS].astype(float)
numeric_mean = numeric.mean()
numeric_std = numeric.std(ddof=0)
assert numeric_std.gt(0).all()
numeric_z = (numeric - numeric_mean) / numeric_std
numeric_z.columns = [f'z__{column}' for column in numeric_z.columns]

dummy_blocks = {}
for column in CATEGORICAL_MODEL_FIELDS:
    values = semantic[column].astype(str)
    block = pd.get_dummies(values, prefix=column, prefix_sep='__', dtype=float)
    block = block.reindex(sorted(block.columns), axis=1)
    assert block.sum(axis=1).eq(1).all()
    dummy_blocks[column] = block

one_hot = pd.concat(dummy_blocks.values(), axis=1)
assert one_hot.shape[1] == 44
assert not one_hot.columns.duplicated().any()
level_retention_audit = pd.DataFrame([
    {
        'field': column,
        'night_levels_n': int(night[column].nunique(dropna=True)),
        'prepared_levels_n': int(prepared[column].nunique(dropna=True)),
        'levels_absent_after_confirmed_exclusions': sorted(
            set(night[column].dropna().astype(int)) - set(prepared[column].dropna().astype(int))
        ),
    }
    for column in CATEGORICAL_MODEL_FIELDS
])
display(level_retention_audit)
unweighted_features = pd.concat([numeric_z, one_hot], axis=1)

weighted_blocks = {
    column: block / np.sqrt(block.shape[1])
    for column, block in dummy_blocks.items()
}
block_weighted_features = pd.concat([numeric_z, *weighted_blocks.values()], axis=1)
assert list(block_weighted_features.columns) == list(unweighted_features.columns)

matrix_unweighted = pd.concat(
    [semantic[['record_id']].reset_index(drop=True), unweighted_features.reset_index(drop=True)], axis=1
)
matrix_block_weighted = pd.concat(
    [semantic[['record_id']].reset_index(drop=True), block_weighted_features.reset_index(drop=True)], axis=1
)
for matrix in [matrix_unweighted, matrix_block_weighted]:
    assert matrix.shape == (6701, 50)
    assert matrix['record_id'].is_unique
    assert not matrix.isna().any().any()
    assert not any(token in column.lower() for column in matrix.columns
                   for token in ['light', 'fatal', 'death', 'injury', 'district', 'longitude', 'latitude'])

encoding_audit = pd.DataFrame([
    {'field': column, 'levels': block.shape[1],
     'unweighted_row_sum': 1.0, 'block_weighted_row_sum': 1 / np.sqrt(block.shape[1])}
    for column, block in dummy_blocks.items()
])
display(encoding_audit)
print({'rows': len(semantic), 'semantic_columns': semantic.shape[1],
       'one_hot_columns': one_hot.shape[1], 'matrix_columns_including_record_id': 50,
       'numeric_means_before_scaling': numeric_mean.to_dict(),
       'numeric_std_ddof0_before_scaling': numeric_std.to_dict()})


,field,night_levels_n,prepared_levels_n,levels_absent_after_confirmed_exclusions
0,weather_code,4,4,[]
1,road_class_code,5,5,[]
2,road_form_code,14,13,[19]
3,crash_position_code,23,22,[25]


,field,levels,unweighted_row_sum,block_weighted_row_sum
0,weather_code,4,1.0,0.500000
1,road_class_code,5,1.0,0.447214
2,road_form_code,13,1.0,0.277350
3,crash_position_code,22,1.0,0.213201


{'rows': 6701, 'semantic_columns': 11, 'one_hot_columns': 44, 'matrix_columns_including_record_id': 50, 'numeric_means_before_scaling': {'speed_limit': 45.388001790777494, 'hour_sin': -0.630600526301313, 'hour_cos': 0.4324560685650953, 'month_sin': -0.07050640367200199, 'month_cos': 0.14549433899729733}, 'numeric_std_ddof0_before_scaling': {'speed_limit': 8.260434022185693, 'hour_sin': 0.5123602560614148, 'hour_cos': 0.3909113620738272, 'month_sin': 0.7125140090594454, 'month_cos': 0.6827767067313979}}


In [9]:
outputs = {
    'nighttime_clustering_input': semantic,
    'matrix_unweighted': matrix_unweighted,
    'matrix_block_weighted': matrix_block_weighted,
}
assert set(outputs) == set(OUTPUTS)

verification = {}
for key, frame in outputs.items():
    path = output_paths[key]
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        existing = pd.read_csv(path, encoding='utf-8-sig')
        pd.testing.assert_frame_equal(
            frame.reset_index(drop=True), existing, check_dtype=False,
            check_exact=False, rtol=1e-12, atol=1e-12,
            obj=f'existing output {key}',
        )
        write_status = 'existing bytes retained because values match regenerated output'
    else:
        with path.open('x', encoding='utf-8-sig', newline='') as stream:
            frame.to_csv(stream, index=False, lineterminator='\n', float_format='%.17g')
        write_status = 'new file written exclusively'

    reread = pd.read_csv(path, encoding='utf-8-sig')
    pd.testing.assert_frame_equal(
        frame.reset_index(drop=True), reread, check_dtype=False,
        check_exact=False, rtol=1e-12, atol=1e-12, obj=f'read-back {key}',
    )
    assert reread['record_id'].is_unique
    assert not reread.isna().any().any()
    verification[key] = {
        'data_path': OUTPUTS[key], 'rows': len(reread), 'columns': len(reread.columns),
        'sha256': sha256_file(path), 'status': f'{write_status}; read-back checks passed',
    }
    print(verification[key])

assert sha256_file(input_path) == actual_sha256, '執行期間輸入檔發生變動。'


{'data_path': 'data/processed/taipei_2025_nighttime_clustering_input.csv', 'rows': 6701, 'columns': 11, 'sha256': '02869cf20f9362a6bf3d54ca2ca8e4b75dca983069d220bc527d6dbe9122b0f1', 'status': 'existing bytes retained because values match regenerated output; read-back checks passed'}
{'data_path': 'data/processed/taipei_2025_nighttime_clustering_matrix_unweighted.csv', 'rows': 6701, 'columns': 50, 'sha256': '6a6d92299d333cce4dcc9b1f425faa28e0d8a20ba2b76dc7221ecc1d85328ab6', 'status': 'existing bytes retained because values match regenerated output; read-back checks passed'}
{'data_path': 'data/processed/taipei_2025_nighttime_clustering_matrix_block_weighted.csv', 'rows': 6701, 'columns': 50, 'sha256': '88173998c8c597af0ce653e1b6764b03ed2582cdb961c3975de12079ab304e37', 'status': 'existing bytes retained because values match regenerated output; read-back checks passed'}
